In [ ]:
!pip install transformers langchain langchain-community langchain-text-splitters pypdf faiss-cpu torch accelerate sentence-transformers pdfplumber
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently t

/tmp/ipykernel_792/368646710.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

In [ ]:
loader=PyPDFLoader('/content/agronomy-book.pdf')
doc=loader.load()

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,

)


In [ ]:
embedding=HuggingFaceEmbeddings(
   model_name='sentence-transformers/all-MiniLM-L6-v2'

)

/tmp/ipykernel_792/1017570282.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector=FAISS.from_documents(
    documents=doc,
    embedding=embedding
)

In [ ]:
vector.save_local('/content/faiss_index')

In [ ]:
from huggingface_hub import login


In [ ]:
model="meta-llama/Llama-3.2-1B-Instruct"
token=AutoTokenizer.from_pretrained(model)
model=AutoModelForCausalLM.from_pretrained(model,
                                           torch_dtype=torch.float,
                                           device_map='auto')

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
generate=pipeline(
    task='text-generation',
    model=model,
    tokenizer=token,
    max_length=1000,
    do_sample=True,
    top_k=10,
)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def ask_pdf(question):
    retrieved_docs = vector.similarity_search(
        question,
        k=3
    )

    context = "\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generate(
        prompt,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.3
    )

    return response[0]["generated_text"]

In [ ]:
t

In [ ]:
while True:
    query = input("Ask: ")

    if query.lower() == "exit":
        break

    answer = ask_pdf(query)
    print(answer)

Ask: what is agriculture


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using the context below.

Context:
2 A TEXTBOOK OF AGRONOMY
land, osier land, market gardens and nursery grounds, and the use of land for woodlands where that use
ancillary to the farming of land for Agricultural purposes”. It is also defined as ‘purposeful work
through which elements in nature are harnessed to produce plants and animals to meet the human
needs. It is a biological production process, which depends on the growth and development of selected
plants and animals within the local environment.
C.  Agriculture as art, science and business of crop production
Agriculture is defined as the art, the science and the business of producing crops and the livestock for
economic purposes.
As an art,  it embraces knowledge of the way to perform the operations of the farm in a skillful
manner. The skill is categorized as;
Physical skill: It involves the ability and capacity to carry out the operation in an efficient way for
e.g., handling of farm implements, animals e

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question using the context below.

Context:
516 A TEXTBOOK OF AGRONOMY
II. Principles Involved-Rice
(i) Threshing: Involves the detachment of grains from the panicle.
(ii) Drying: Reduction of 12–14% or 8% by evaporation. i.e., it involves heat and mass transfer
operations simultaneously.
(iii) Parboiling: Is a hydrothermal treatment followed by drying before milling for the production of
milled parboiled grain. The most important change during parboiling is the gelatinization of
starch and disintegration of protein bodies in the endosperm.
(iv) Milling: Refers to the size reduction and separation operations used for processing of food
grains into edible form by removing and separating the inedible and undesirable portions from
them, Milling may involve cleaning/separating husk (dehusking), sorting, whitening, polishing,
grinding etc.
(v) Storage: Proper storage in storage structures is necessary to prevent the grains from storage pest
and to maintain the quality of seeds.


KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install flask pyngrok -q


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token('3EdXP7om0eRJz7FpBFd50y2kOOp_87z1qW3UWDhrTjeVKiKwg')

In [ ]:
from flask import Flask, request, jsonify
import threading

# LLaMA 3.2 1B — lightweight!
llm = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

app = Flask(__name__)

@app.route("/ask", methods=["POST"])
def ask():
    question = request.json["question"]

    messages = [{"role": "user", "content": question}]

    output = llm(
        messages,
        max_new_tokens=150,
        do_sample=False
    )

    answer = output[0]["generated_text"][-1]["content"]
    return jsonify({"answer": answer})

public_url = ngrok.connect(5000)
print("🌐 URL:", public_url)

threading.Thread(target=app.run,
                 kwargs={"port": 5000}).start()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

🌐 URL: NgrokTunnel: "https://goofball-confound-flaring.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import requests

url = "https://goofball-confound-flaring.ngrok-free.dev/ask"  # Updated URL to include /ask endpoint
res = requests.post(url, json={"question": "What is agriculture?"})
print(res.json())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 04:22:26] "POST /ask HTTP/1.1" 200 -


{'answer': 'Agriculture is the practice of cultivating and producing food, as well as other crops, for human consumption, animal feed, and other uses. It involves the use of land, water, and other natural resources to grow and harvest crops, raise livestock, and manage forests and other ecosystems.\n\nAgriculture can be divided into several subcategories, including:\n\n1. **Food agriculture**: This includes crops such as fruits, vegetables, grains, and legumes, as well as livestock such as cattle, pigs, and chickens.\n2. **Animal agriculture**: This includes the production of meat, dairy products, and other animal-derived foods.\n3. **Horticulture**: This includes the cultivation of fruits, vegetables, and flowers for human consumption.\n4'}
